### Needs `01-Prep-BaselinePipelineAB.ipynb` to be executed first

Reads from:
- `Baseline-PipelineA-PipelineB_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Import Data

In [2]:
answers = pd.read_json("Baseline-PipelineA-PipelineB_ynorm.json")

CATEGORIES = ["value", "unit"]
VARIANTS = ["Base", "A", "B"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()

    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()

    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [5]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,          # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)

    return result

def eval(source, exact) -> pd.DataFrame:

    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)

    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)

    return gs_eval, in_place



def print_eval(gs_eval) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<4}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [6]:
ynorm_exact, ynorm_exact_extended = eval(answers, True)

print_eval(ynorm_exact)

### value ###
Base: True = 2073 || False = 135 || Quota = 93.89%
A   : True = 2100 || False = 108 || Quota = 95.11%
B   : True = 2124 || False = 84 || Quota = 96.2%

### unit ###
Base: True = 1983 || False = 225 || Quota = 89.81%
A   : True = 1967 || False = 241 || Quota = 89.09%
B   : True = 1995 || False = 213 || Quota = 90.35%



In [7]:
ynorm_exact_any, ynorm_exact_extended_any = eval(answers, False)

print_eval(ynorm_exact_any)

### value ###
Base: True = 2090 || False = 118 || Quota = 94.66%
A   : True = 2108 || False = 100 || Quota = 95.47%
B   : True = 2139 || False = 69 || Quota = 96.88%

### unit ###
Base: True = 1991 || False = 217 || Quota = 90.17%
A   : True = 1967 || False = 241 || Quota = 89.09%
B   : True = 1998 || False = 210 || Quota = 90.49%



In [8]:
ynorm_exact_extended.to_csv("Baseline-PipelineA-PipelineB-Eval.csv")

## By Scope

In [9]:
# Scope-Aufschlüsselung — in jedes 02-Eval-...ipynb einfügbar, direkt nach print_eval()
def print_eval_by_scope(gs_eval) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for scope in sorted(gs_eval["scope"].unique()):
            sub = gs_eval[gs_eval["scope"] == scope]
            print(f"  -- scope {scope} --")
            for v in VARIANTS:
                col = sub[f"{v}_{cat}_hit"]
                true_count  = col.value_counts().get(True, 0)
                false_count = col.value_counts().get(False, 0)
                quota       = round(col.mean() * 100, 2)
                print(f"  {v:<4}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

print_eval_by_scope(ynorm_exact_any)

### value ###
  -- scope 1 --
  Base: True = 542 || False = 19 || Quota = 96.61%
  A   : True = 543 || False = 18 || Quota = 96.79%
  B   : True = 547 || False = 14 || Quota = 97.5%
  -- scope 2lb --
  Base: True = 512 || False = 39 || Quota = 92.92%
  A   : True = 524 || False = 27 || Quota = 95.1%
  B   : True = 542 || False = 9 || Quota = 98.37%
  -- scope 2mb --
  Base: True = 517 || False = 28 || Quota = 94.86%
  A   : True = 525 || False = 20 || Quota = 96.33%
  B   : True = 533 || False = 12 || Quota = 97.8%
  -- scope 3 --
  Base: True = 519 || False = 32 || Quota = 94.19%
  A   : True = 516 || False = 35 || Quota = 93.65%
  B   : True = 517 || False = 34 || Quota = 93.83%

### unit ###
  -- scope 1 --
  Base: True = 494 || False = 67 || Quota = 88.06%
  A   : True = 478 || False = 83 || Quota = 85.2%
  B   : True = 493 || False = 68 || Quota = 87.88%
  -- scope 2lb --
  Base: True = 484 || False = 67 || Quota = 87.84%
  A   : True = 477 || False = 74 || Quota = 86.57%
  B   : 